# Run Spatial Fluid-LeWM

Use this notebook from the repo root in your `data-monolith` environment. It wraps the new Hydra model variants so you can launch training and evaluation without reconstructing the CLI manually.


In [28]:
from datetime import datetime
import importlib
from pathlib import Path
import os
import shlex
import subprocess
import sys

os.environ.setdefault("HYDRA_FULL_ERROR", "1")

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / "train.py").exists(), "Open the notebook from the repo root."

print("Repo root:", REPO_ROOT)
print("Python:", sys.executable)
print("STABLEWM_HOME:", os.environ.get("STABLEWM_HOME", str(Path.home() / ".stable-wm")))


Repo root: /home/mengyuwsl/leworldmodel/le-wm
Python: /home/mengyuwsl/ds-monolith/bin/python
STABLEWM_HOME: /home/mengyuwsl/.stable-wm


In [29]:
REQUIRED_IMPORTS = {
    "torch": "torch",
    "hydra": "hydra-core",
    "omegaconf": "omegaconf",
    "lightning": "lightning",
    "einops": "einops",
    "wandb": "wandb",
    "stable_worldmodel": "stable-worldmodel[train,env]",
    "stable_pretraining": "stable-pretraining",
}

missing = []
for module_name, package_name in REQUIRED_IMPORTS.items():
    try:
        importlib.import_module(module_name)
        print(f"ok: {module_name}")
    except ModuleNotFoundError:
        missing.append((module_name, package_name))
        print(f"missing: {module_name}  ->  install package: {package_name}")

if missing:
    install_targets = " ".join(sorted({pkg for _, pkg in missing}))
    raise RuntimeError(
        "Missing notebook/runtime dependencies. Install them in data-monolith, for example:\n"
        f"python -m pip install {install_targets}"
    )


ok: torch
ok: hydra
ok: omegaconf
ok: lightning
ok: einops
ok: wandb
ok: stable_worldmodel
ok: stable_pretraining


In [30]:
# Select the training/eval run you want.
# DATASET: pusht | tworoom | ogb | dmc
# MODEL: cls_transformer | spatial_transformer | spatial_pde
# EVAL_CONFIG: pusht | tworoom | cube | reacher

DATASET = "pusht"
MODEL = "spatial_pde"
EVAL_CONFIG = "pusht"
STABLEWM_HOME = os.environ.get("STABLEWM_HOME", str(Path.home() / ".stable_worldmodel"))
WANDB_ENABLED = False
EXTRA_TRAIN_OVERRIDES = []
# Example debug fallback if rank metrics still cause trouble:
# EXTRA_TRAIN_OVERRIDES = ["wm.log_spatial_metrics=False"]
EXTRA_EVAL_OVERRIDES = []

RUN_STEM = f"notebooks/{MODEL}_{DATASET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_MODEL_NAME = {
    "cls_transformer": "lewm_cls",
    "spatial_transformer": "spatial_lewm_transformer",
    "spatial_pde": "spatial_lewm_pde",
}[MODEL]

print("Run stem:", RUN_STEM)
print("Output model name:", OUTPUT_MODEL_NAME)
print("STABLEWM_HOME:", STABLEWM_HOME)


Run stem: notebooks/spatial_pde_pusht_20260327_150021
Output model name: spatial_lewm_pde
STABLEWM_HOME: /home/mengyuwsl/.stable_worldmodel


In [31]:
def stablewm_home() -> Path:
    return Path(STABLEWM_HOME).expanduser()


def run_cmd(cmd, cwd=REPO_ROOT):
    print("+", shlex.join([str(part) for part in cmd]))
    env = os.environ.copy()
    env["STABLEWM_HOME"] = str(stablewm_home())
    env.setdefault("PYTHONUNBUFFERED", "1")
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(lines[-40:])
        raise RuntimeError(f"Command failed with exit code {return_code}\nLast output:\n{tail}")


def latest_policy_path(run_stem: str, output_model_name: str) -> str:
    run_dir = stablewm_home() / run_stem
    ckpts = sorted(run_dir.glob(f"{output_model_name}_epoch_*_object.ckpt"))
    if not ckpts:
        raise FileNotFoundError(f"No object checkpoints found in {run_dir}")
    latest = max(ckpts, key=lambda path: path.stat().st_mtime)
    return latest.relative_to(stablewm_home()).as_posix().removesuffix("_object.ckpt")


In [32]:
EXPECTED_DATASETS = {
    "pusht": ["pusht_expert_train.h5"],
    "tworoom": ["tworoom.h5"],
    "ogb": ["ogbench/cube_single_expert.h5", "ogbench_cube_single_expert.h5"],
    "dmc": ["reacher.h5", "dmc/reacher.h5", "dmc_reacher_random.h5"],
}

dataset_root = stablewm_home()
print("Dataset root:", dataset_root)
candidates = [dataset_root / rel for rel in EXPECTED_DATASETS.get(DATASET, [])]
for path in candidates:
    print("check:", path)

if candidates and not any(path.exists() for path in candidates):
    raise FileNotFoundError(
        "Dataset file not found for the selected DATASET. "
        f"Set STABLEWM_HOME to the directory containing the .h5 files. Checked: {candidates}"
    )


Dataset root: /home/mengyuwsl/.stable_worldmodel
check: /home/mengyuwsl/.stable_worldmodel/pusht_expert_train.h5


FileNotFoundError: Dataset file not found for the selected DATASET. Set STABLEWM_HOME to the directory containing the .h5 files. Checked: [PosixPath('/home/mengyuwsl/.stable_worldmodel/pusht_expert_train.h5')]

In [ ]:
train_cmd = [
    sys.executable,
    "train.py",
    f"data={DATASET}",
    f"model={MODEL}",
    f"subdir={RUN_STEM}",
    f"wandb.enabled={str(WANDB_ENABLED)}",
] + EXTRA_TRAIN_OVERRIDES

print(shlex.join([str(part) for part in train_cmd]))


/home/mengyuwsl/ds-monolith/bin/python train.py data=pusht model=spatial_pde subdir=notebooks/spatial_pde_pusht_20260327_144936 wandb.enabled=False


In [ ]:
# Run training.
run_cmd(train_cmd)

POLICY = latest_policy_path(RUN_STEM, OUTPUT_MODEL_NAME)
print("Latest policy:", POLICY)


+ /home/mengyuwsl/ds-monolith/bin/python train.py data=pusht model=spatial_pde subdir=notebooks/spatial_pde_pusht_20260327_144936 wandb.enabled=False
Error executing job with overrides: ['data=pusht', 'model=spatial_pde', 'subdir=notebooks/spatial_pde_pusht_20260327_144936', 'wandb.enabled=False']
14:49:48.170 | ERROR   (138806, richuru) | Exception:
Traceback (most recent call last):

> File "/home/mengyuwsl/leworldmodel/le-wm/train.py", line 318, in <module>
    run()
    └ <function run at 0x7be5904bc900>

  File "/home/mengyuwsl/ds-monolith/lib/python3.12/site-packages/hydra/main.py", line 94, in decorated_main
    _run_hydra(
    └ <function _run_hydra at 0x7be72b8e3920>
  File "/home/mengyuwsl/ds-monolith/lib/python3.12/site-packages/hydra/_internal/utils.py", line 394, in _run_hydra
    _run_app(
    └ <function _run_app at 0x7be72b8e39c0>
  File "/home/mengyuwsl/ds-monolith/lib/python3.12/site-packages/hydra/_internal/utils.py", line 457, in _run_app
    run_and_report(
    └ <

RuntimeError: Command failed with exit code 1
Last output:
    ret.return_value = task_function(task_cfg)
    │   │              │             └ {'output_model_name': 'lewm', 'subdir': 'notebooks/spatial_pde_pusht_20260327_144936', 'num_workers': 6, 'train_split': 0.9, ...
    │   │              └ <function run at 0x7be5904bc860>
    │   └ <property object at 0x7be72b8a2110>
    └ JobReturn(overrides=['data=pusht', 'model=spatial_pde', 'subdir=notebooks/spatial_pde_pusht_20260327_144936', 'wandb.enabled=...

  File "/home/mengyuwsl/leworldmodel/le-wm/train.py", line 217, in run
    dataset = swm.data.HDF5Dataset(**cfg.data.dataset, transform=None)
              │   │    │             └ {'output_model_name': 'lewm', 'subdir': 'notebooks/spatial_pde_pusht_20260327_144936', 'num_workers': 6, 'train_split': 0.9, ...
              │   │    └ <class 'stable_worldmodel.data.dataset.HDF5Dataset'>
              │   └ <module 'stable_worldmodel.data' from '/home/mengyuwsl/ds-monolith/lib/python3

In [ ]:
POLICY = latest_policy_path(RUN_STEM, OUTPUT_MODEL_NAME)
eval_cmd = [
    sys.executable,
    "eval.py",
    f"--config-name={EVAL_CONFIG}.yaml",
    f"policy={POLICY}",
] + EXTRA_EVAL_OVERRIDES

print(shlex.join([str(part) for part in eval_cmd]))


In [ ]:
# Run evaluation after training.
run_cmd(eval_cmd)
